In [ ]:
import numpy as np
import torch
import os
from pathlib import Path
import matplotlib.pyplot as plt
# from data_preparation.field import load_fields

In [ ]:
processed_data_dir = Path("../processed_data")
exp_name = "history1_fv"
case_name = "flange"
case_name = f"{case_name}_{exp_name}"
case_dir = processed_data_dir / case_name
checkpoint_dir = case_dir / "checkpoints"
pred_dir = case_dir / "predictions"
error_dir = case_dir / "errors"

In [ ]:
train_loss = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
val_loss = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
rollout_mae = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, h = 1, FV", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{case_name}.png", dpi=300, transparent = True)

In [ ]:
train_loss, val_loss, rollout_mae = {},{},{}
processed_data_dir = Path("../processed_data")
for case_name in ["flange_history1_fv", "flange_history1_nofv"]:
    case_dir = processed_data_dir / case_name
    checkpoint_dir = case_dir / "checkpoints"
    pred_dir = case_dir / "predictions"
    error_dir = case_dir / "errors"

    train_loss[case_name] = np.load(os.path.join(checkpoint_dir, "train_losses.npy"))
    val_loss[case_name] = np.load(os.path.join(checkpoint_dir, "val_losses.npy"))
    rollout_mae[case_name] = np.load(os.path.join(checkpoint_dir, "rollout_mae.npy"))

In [ ]:
fig, axs = plt.subplots(1,1)
axs.set_title("Training Loss", fontsize=20)
axs.set_yscale("log")
axs.set_xlabel("Epoch", fontsize=18)
axs.set_ylabel("MSE Loss", fontsize=18)
for i,case_name in enumerate(train_loss):
    plt.plot(train_loss[case_name], label=f"{case_name}: Train Loss")
    plt.plot(val_loss[case_name], label=f"{case_name}: Validation Loss")

axs.legend(fontsize=14)

In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

fig, axs = plt.subplots(1,1, figsize=(960*px, 540*px))


# Transparent background
loss_graph.patch.set_alpha(0)
axs.patch.set_alpha(0)

axs.title.set_color(AX_COLOR)
axs.xaxis.label.set_color(AX_COLOR)
axs.yaxis.label.set_color(AX_COLOR)
axs.tick_params(colors=AX_COLOR)
for spine in axs.spines.values():
    spine.set_edgecolor(AX_COLOR)


for i,case_name in enumerate(train_loss):
    if case_name.endswith("_fv"):
        plt.plot(rollout_mae[case_name], label="FV", color = PLOT1_COLOR, linewidth=2.5)
    elif case_name.endswith("_nofv"):
        plt.plot(rollout_mae[case_name], label="No FV", color = PLOT2_COLOR, linewidth=2.5, linestyle='--')

legend = axs.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0–1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

axs.set_title("Rollout Error Accumulation, h = 1", fontsize=24)
axs.set_xlabel("Time step", fontsize=18)
axs.set_ylabel("MAE", fontsize=18)
axs.tick_params(axis='both', labelsize=16)
axs.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.savefig(f"../outputs/plots/{case_name[:15]}_rollout.png", dpi=300, transparent = True)


## Parametric case

Training/validation-loss and rollout-error plots for the parametric runs at `history = 1`.
The rollout panels mirror the flange **FV vs no-FV** comparison
(`parametric_history1_mesh` vs `parametric_history1_mesh_nofv`), one panel per
held-out test mesh (`model_003` and `model_007`).

In [ ]:
# --- Parametric case (history = 1, FV vs no-FV) ------------------------
# Each parametric run is tested on the held-out meshes model_003 and
# model_007, so there is one rollout-MAE curve per test mesh.
param_cases = {
    "fv":   "parametric_history1_mesh",
    "nofv": "parametric_history1_mesh_nofv",
}
param_test_meshes = ["model_003", "model_007"]

# The FV run drives the single-case loss plot below.
param_case = param_cases["fv"]
param_ckpt = processed_data_dir / param_case / "checkpoints"
param_train_loss = np.load(os.path.join(param_ckpt, "train_losses.npy"))
param_val_loss = np.load(os.path.join(param_ckpt, "val_losses.npy"))

# rollout MAE per {fv/nofv} run and per test mesh.
param_rollout = {}
for key, cname in param_cases.items():
    ckpt = processed_data_dir / cname / "checkpoints"
    param_rollout[key] = {
        mesh: np.load(os.path.join(ckpt, f"rollout_mae_{mesh}.npy"))
        for mesh in param_test_meshes
    }


In [ ]:
AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

loss_graph, ax = plt.subplots(figsize=(960*px, 540*px))

# Transparent background
loss_graph.patch.set_alpha(0)
ax.patch.set_alpha(0)

ax.title.set_color(AX_COLOR)
ax.xaxis.label.set_color(AX_COLOR)
ax.yaxis.label.set_color(AX_COLOR)
ax.tick_params(colors=AX_COLOR)
for spine in ax.spines.values():
    spine.set_edgecolor(AX_COLOR)

plt.plot(param_train_loss, label="Train Loss", color = PLOT1_COLOR, linewidth=2.5)
plt.plot(param_val_loss, label="Validation Loss", color = PLOT2_COLOR, linewidth=2.5)
ax.set(yscale='log')

legend = ax.legend(fontsize=18)
legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
legend.get_frame().set_edgecolor(AX_COLOR)        # border color
legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
for text in legend.get_texts():
    text.set_color(AX_COLOR)                      # label text color

plt.xlabel("Epoch")
plt.ylabel("Loss")
ax.set_title("Training and Validation Loss, Parametric FV, h = 1", fontsize=24)
ax.set_yscale("log")
ax.set_xlabel("Epoch", fontsize=18)
ax.set_ylabel("MSE Loss", fontsize=18)
ax.tick_params(axis='both', labelsize=16)
ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)


# plt.savefig(f"../outputs/plots/{param_case}.png", dpi=300, transparent = True)


In [ ]:
from matplotlib.ticker import StrMethodFormatter

AX_COLOR = "#1e2a2c"
LEGEND_BG = "#f0eee9"
PLOT1_COLOR = "#5e3d80"
PLOT2_COLOR = "#174535"

px = 1/plt.rcParams['figure.dpi']  # pixel in inches

# One panel per held-out test mesh; each panel is the flange-style
# FV vs no-FV rollout comparison.
fig, axs = plt.subplots(len(param_test_meshes),1,
                        figsize=(960*px, len(param_test_meshes)*540*px),
                        constrained_layout=True)

# Transparent background
fig.patch.set_alpha(0)

for ax, mesh in zip(axs, param_test_meshes[::-1]):
    ax.patch.set_alpha(0)
    ax.title.set_color(AX_COLOR)
    ax.xaxis.label.set_color(AX_COLOR)
    ax.yaxis.label.set_color(AX_COLOR)
    ax.tick_params(colors=AX_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(AX_COLOR)

    ax.plot(param_rollout["fv"][mesh], label="FV",
            color=PLOT1_COLOR, linewidth=2.5)
    ax.plot(param_rollout["nofv"][mesh], label="No FV",
            color=PLOT2_COLOR, linewidth=2.5, linestyle='--')

    if mesh == "model_003":
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.1e}'))



    legend = ax.legend(fontsize=18)
    legend.get_frame().set_facecolor(LEGEND_BG)   # background fill
    legend.get_frame().set_edgecolor(AX_COLOR)        # border color
    legend.get_frame().set_alpha(1)                # fully opaque (or 0-1)
    for text in legend.get_texts():
        text.set_color(AX_COLOR)                      # label text color

    ax.set_title(f"Rollout Error Accumulation, {mesh}, h = 1", fontsize=24)
    ax.set_xlabel("Time step", fontsize=18)
    ax.set_ylabel("MAE", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, which="both", ls="--", linewidth=0.5, color=AX_COLOR, alpha=0.7)

# plt.tight_layout(pad=0)
# plt.savefig("../outputs/plots/parametric_history1_mesh_rollout_exp.png", dpi=300, transparent = True)